**Objetivo:** ingerir os 5 CSVs brutos (Volume) e a cotação do dólar (API do Banco Central) na camada **Bronze**.

**Regras desta camada**
- Nenhuma alteração estrutural ou de conteúdo: todas as colunas são lidas como `STRING` (sem `inferSchema`).
- Única coluna acrescentada: `ingestion_datetime` (timestamp da inserção).
- Gravação em **Delta** com modo **append** (por isso reexecuções geram linhas repetidas na Bronze; a deduplicação é responsabilidade da Silver).

**Parâmetros (widgets):** `catalog`, `data_inicio`, `data_fim` (formato `MM-DD-AAAA`, exigido pela API do BCB).

In [0]:
import re
from datetime import date, datetime, timedelta

import requests
from pyspark.sql import functions as F

# Formato de data exigido pela API do Banco Central: MM-DD-AAAA
FMT_API = "%m-%d-%Y"
hoje = date.today()

# Widgets: por padrão consultam os últimos 7 dias corridos, pois a API não
# retorna cotação em finais de semana e feriados (a janela garante dias úteis).
dbutils.widgets.text("catalog", "workspace", "Catálogo (Unity Catalog)")
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=7)).strftime(FMT_API), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", hoje.strftime(FMT_API), "Data fim (MM-DD-AAAA)")

catalog = dbutils.widgets.get("catalog").strip()
data_inicio = dbutils.widgets.get("data_inicio").strip()
data_fim = dbutils.widgets.get("data_fim").strip()

# Falha cedo (com mensagem clara) se as datas estiverem no formato errado
for nome, valor in [("data_inicio", data_inicio), ("data_fim", data_fim)]:
    try:
        datetime.strptime(valor, FMT_API)
    except ValueError:
        raise ValueError(f"Widget '{nome}' inválido: '{valor}'. Use o formato MM-DD-AAAA.")

print(f"Catálogo: {catalog} | Período da cotação: {data_inicio} → {data_fim}")

Catálogo: workspace | Período da cotação: 09-14-2026 → 09-21-2026


In [0]:
spark.sql(f"USE CATALOG `{catalog}`")

# Schema/Volume de origem (o upload dos CSVs é feito uma única vez pela interface do Databricks)
spark.sql("CREATE SCHEMA IF NOT EXISTS landing")
spark.sql("CREATE VOLUME IF NOT EXISTS landing.inputs")

# Banco de dados (schema) da camada Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

VOLUME_PATH = f"/Volumes/workspace/default/inputs/"
print("Arquivos encontrados no Volume:")
for f in dbutils.fs.ls(VOLUME_PATH):
    print(" -", f.name)

Arquivos encontrados no Volume:
 - credits_and_tags_IMDB_TMDB.csv
 - movies_financials_IMDB_TMDB.csv
 - movies_info_TMDB_IMDB.csv
 - movies_metrics_IMDB_TMDB.csv
 - movies_reviews.csv


In [0]:
#Leitura com `multiLine` + `escape` porque sinopses e comentários podem conter quebras de linha e aspas.
#Todas as colunas permanecem `STRING` para preservar o dado bruto (inclusive a sujeira proposital).

ARQUIVOS = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}

# Caracteres que o Delta não aceita em nomes de coluna (espaço , ; { } ( ) \n \t =).
# Só mexemos no NOME da coluna se for tecnicamente inevitável; o conteúdo nunca é alterado.
COLUNAS_INVALIDAS = r"[ ,;{}()\n\t=]"


def sanitizar_colunas(df, nome_tabela):
    novas = [re.sub(COLUNAS_INVALIDAS, "_", c.strip()) for c in df.columns]
    if novas != df.columns:
        print(f"  [aviso] {nome_tabela}: nomes de coluna ajustados para o Delta -> {dict(zip(df.columns, novas))}")
    return df.toDF(*novas)


def ingerir_csv(nome_arquivo, tabela):
    caminho = f"{VOLUME_PATH}/{nome_arquivo}"
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", False)   # tudo STRING: nenhuma alteração de conteúdo
        .option("multiLine", True)      # textos com quebra de linha
        .option("quote", '"')
        .option("escape", '"')
        .load(caminho)
    )
    df = sanitizar_colunas(df, tabela)

    # Timestamp exato da inserção na Bronze (mesmo valor para todas as linhas do lote)
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    (df.write.format("delta").mode("append").saveAsTable(tabela))
    print(f"[ok] {nome_arquivo} -> {tabela} | colunas: {len(df.columns) - 1}")


for arquivo, tabela in ARQUIVOS.items():
    ingerir_csv(arquivo, tabela)

[ok] movies_info_TMDB_IMDB.csv -> bronze.tb_movies_info | colunas: 10
[ok] movies_financials_IMDB_TMDB.csv -> bronze.tb_movies_financials | colunas: 3
[ok] movies_metrics_IMDB_TMDB.csv -> bronze.tb_movies_metrics | colunas: 6
[ok] credits_and_tags_IMDB_TMDB.csv -> bronze.tb_credits_and_tags | colunas: 9
[ok] movies_reviews.csv -> bronze.tb_movies_reviews | colunas: 4


In [0]:
URL = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)


def consultar_api(url, tentativas=3, espera=5):
    """Chama a API com retentativas simples (instabilidade de rede é comum)."""
    import time
    ultimo_erro = None
    for i in range(1, tentativas + 1):
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            return resp.json().get("value", [])
        except Exception as e:
            ultimo_erro = e
            print(f"  tentativa {i}/{tentativas} falhou: {e}")
            time.sleep(espera)
    raise RuntimeError(f"Não foi possível consultar a API do BCB: {ultimo_erro}")


cotacoes = consultar_api(URL)
print(f"Registros retornados pela API: {len(cotacoes)}")
if not cotacoes:
    print("[aviso] A API não retornou cotações para o período informado (amplie a janela de datas).")

SCHEMA_COTACAO = "dataHoraCotacao STRING, cotacaoCompra DOUBLE"
df_cotacao = spark.createDataFrame(
    [(c["dataHoraCotacao"], float(c["cotacaoCompra"])) for c in cotacoes],
    SCHEMA_COTACAO,
).withColumn("ingestion_datetime", F.current_timestamp())

(df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar"))
print("[ok] API BCB -> bronze.tb_cotacao_dolar")

Registros retornados pela API: 6
[ok] API BCB -> bronze.tb_cotacao_dolar


In [0]:
tabelas_bronze = list(ARQUIVOS.values()) + ["bronze.tb_cotacao_dolar"]
for t in tabelas_bronze:
    df = spark.table(t)
    assert "ingestion_datetime" in df.columns, f"{t} sem ingestion_datetime"
    print(f"{t:<35} linhas: {df.count():>9,}  colunas: {len(df.columns)}")

display(spark.table("bronze.tb_cotacao_dolar").orderBy(F.col("dataHoraCotacao").desc()).limit(10))

bronze.tb_movies_info               linhas:   427,386  colunas: 11
bronze.tb_movies_financials         linhas:   424,660  colunas: 4
bronze.tb_movies_metrics            linhas:   425,164  colunas: 7
bronze.tb_credits_and_tags          linhas:   424,568  colunas: 10
bronze.tb_movies_reviews            linhas:   129,648  colunas: 5
bronze.tb_cotacao_dolar             linhas:        23  colunas: 3


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-21 13:06:51.445645,5.1111,2026-09-21T18:12:39.154Z
2026-09-21 13:06:51.445645,5.1111,2026-09-21T18:31:58.545Z
2026-09-21 13:06:51.445645,5.1111,2026-09-21T17:22:53.722Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T17:22:53.722Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T14:55:50.971Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T18:31:58.545Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T18:12:39.154Z
2026-09-17 13:03:21.858212,5.1515,2026-09-21T17:22:53.722Z
2026-09-17 13:03:21.858212,5.1515,2026-09-21T18:31:58.545Z
2026-09-17 13:03:21.858212,5.1515,2026-09-21T18:12:39.154Z
